# Machine Learning for Chemistry

A key challenge for machine learning is chemistry is to convert molecular structures into formats that can be understood by computers. Chemists typically convey information about molecular structure through skeletal drawings that include a lot of implicit information, such as:
* Atomic composition
* Atomic connectivity
* Stereochemistry 
* Bond orders
* Formal charge

This information is readily understood by humans but can be tricky to encode in computer-friendly representations. 

# Molecular Representations

#### SMILES
SMILES (simplified molecular input entry line system) strings are text-based representations of molecular structure that are commonly used to represent molecules in various databases. They provide a complete picture of the connectivity of a molecule and can be used to encode *some* spatial orientation of atoms. 

The image below shows how a SMILES string is related to skeletal molecular structure. One structure can be represented by many different SMILES strings, meaning  they are not *invariant* representations. This is an important consideration for machine learning, as two SMILES strings for the same molecule could produce different outputs if used as the input for a predictive model. Despite this, some generative models can be trained to produce SMILES strings for molecular discovery. 

![smiles](https://upload.wikimedia.org/wikipedia/commons/0/00/SMILES.png)

Use the visualiser code below to inspect some of the SMILES strings for common drugs:
* Morphine: CN1CC[C@@]23[C@H]4OC5=C2C(C[C@@H]1[C@@H]3C=C[C@@H]4O)=CC=C5O
* Ibuprofen:  CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O
* Penicillin: CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C



In [ ]:
from ipywidgets import interact

from rdkit import Chem
from rdkit.Chem import AllChem

import py3Dmol

smiles = "CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C" # place smiles here

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

AllChem.EmbedMolecule(mol)
AllChem.MMFFOptimizeMolecule(mol)
mol_block = Chem.MolToMolBlock(mol)

port = py3Dmol.view()
port.addModel(mol_block)
port.setStyle({'model': -1}, {"stick": {'radius': 0.15}, "sphere": {"colorscheme": "Jmol", 'radius': .4}})
port.zoomTo()
port.show()


#### Molecular Fingerprints

An alternative molecular representation is a *molecular fingerprint*, a binary array that describes the substructures present within a molecule. 

There are different ways of generating molecular fingerprints. the simplest is to use a pre-defined list of substructures ([MACCS implementation](https://github.com/rdkit/rdkit/blob/master/rdkit/Chem/MACCSkeys.py)), but this can miss important atoms. An alternative is to construct Extended Connectivity FingerPrints (ECFPs) which account for the local environment of each atom and use a hashing function to encode their positions within the fingerprint array. 

Unlike with SMILEs strings, a molecular fingerprint cannot be used to derive it molecular structure.  Another disadvantage of ECFPs is the tendency for certain molecular environments to be hashed down to the same bit in a binary fingerprint array. These are called bit collisions. 

----

# Solubility Task

For a molecule to act effectively as a drug, it must be soluble under physiological conditions. Therefore, the solubility of a compound must be known before it is assessed as a drug candidate. We can use machine learning for such tasks to try and predict the solubility of a drug based on its molecular structure. 

For this task, we will train a machine learning model to predict a molecules solubility from its molecular structure using ECFPs. The code below loads in the data from the [Therapeutics Data Commons](https://tdc.readthedocs.io/en/main/) ADME library. 

You will need to use the following functions from RDKit, a python library specifically designed for cheminformatics:

* `mol = Chem.MolFromSmiles(smiles) # loads in a SMILES string and converts it to and RDKit molecule object`
* `generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048) # sets up fp generator`
* `fp = np.array(generator.GetFingerptint(mol)) # generate the fingerprint and convert to an np array`

These have been imported for you.

You will need to:
* Write a function to convert smiles strings into molecular fingerprints using RDKit
* Train an ML model to predict molecular solubility from its ECFP with the highest possible accuracy


In [ ]:
from tdc.single_pred.adme import ADME
import pandas as pd
import numpy as np

# rdkit imports
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

solubility_data = ADME('Solubility_AqSolDB')
solubility_split = solubility_data.get_split() # use predefined splits 

train_df = pd.concat([solubility_split["train"], solubility_split["valid"] ])
test_df = solubility_split["test"]

# convert dataframes to np arrays 
train_smiles = train_df["Drug"].to_numpy()
train_solubility = train_df["Y"].to_numpy()

test_smiles = test_df["Drug"].to_numpy()
test_solubility = test_df["Y"].to_numpy()

... # your code here

# Drug Design Properties

Solubility is not the only molecular property that is important for drug discovery. There are many more factors to consider, including those that make up the ADMET properties:
* Adsorption
* Distribution 
* Metabolism
* Excretion
* Toxicity

A molecule must also provide some therapeutic effect against some biological target for it to be an effective drug. The Activity of a drug is usually measured experimentally using biological assays. This can be done at large scales using high-throughput screening, where hundreds of potential drug compounds are tested against a relevant target at a time. 

A molecules activity can be measured as either of the following:
* $IC_{50}$: concentration at which half of the target is inhibited by the drug compound
* $K_i$: the equilibrium constant for drug binding.

The lower these values, the more effective the drug. 

----

# Virtual Screening Activity

It can be very costly and time-consuming to experimentally verify every potential drug compound, especially for a newly identified target that has little available activity data. 

If we train a machine learning model to predict biological activity from molecular structure, we can virtually screen a lot of molecules to find the most promising structures to investigate experimentally. 

You are provided with a set of activity data for a biological target of interest `activity_data.csv` and a list of potential drug candidates `drug_candidates.csv`.

 For this task:
 * Use the activity data to train an ML model
 * Find the top 100 drug candidates that should be tested experimentally. 
 * Store the SMILES string and the ML prediction in a pandas dataframe under columns "SMILES" and "predicted_activity". We will use this in the next section. 




In [ ]:
... # your code here

----
# Molecular Similarity

Now we know which 100 compounds we predict to be the most promising drug candidates, it would be interesting to see what molecular features are the most important for reaching high activity. We *could* visualise our top 5 compounds and manually inspect how similar they are, but we may miss other important information that is included in our top 100 molecules. 

There are other ways we can calculate molecular similarity and visualise the overlap of molecular features in our dataset. 

#### Tanimoto Similarity
The Tanimoto similarity coefficient is widely used to compare the similarity of two binary arrays. It gives a value between 0 and 1, and can be applied to our molecular fingerprints to measure molecular similarity. 

It is calculated by considering the number of "on" bits in two fingerprints (eg/ bits that are 1 instead of 0):
$$
T(A,B) = \frac{c}{a+b-c}
$$

where:
* a = no. of "on" bits in A
* b = no. of "on" bits in B
* c = no. of "on" bits in bot A and B

If we compare every molecule in our dataset to every other molecule, we can get an overall picture of the molecular diversity. 

Use the functions below to plot a heatmap of the pairwise Tanimoto similarities for your 100 top molecules 


In [ ]:
import numpy as np
import pandas as pd

from rdkit import DataStructs, Chem
from rdkit.Chem import rdFingerprintGenerator
import matplotlib.pyplot as plt

def get_fingerprints(smiles):
    """
    Converts a pd.Series of SMILES strings to a list of Fingerprints
    :param smiles: pd.Series of SMILES strings
    :return: 
        fps: List of fingerprints
    """
    mols = [Chem.MolFromSmiles(smi) for smi in smiles]
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    return [generator.GetFingerprint(m) for m in mols]


def calculate_pairwise_similarity(df):
    """
    Calculates the pairwise Tanimoto Similarity matrix from the provided dataframe
    :param df: pd.DataFrame containing column 'SMILES' that contains SMILES strings for molecules
    :return: np.ndarray, pairwise Tanimoto similarity matrix
    """
    # get rdmols 
    fps = get_fingerprints(df["SMILES"])
        
    similarity_matrix = np.array([
        DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps
    ])
    
    return similarity_matrix

def plot_similarity_heatmap(similarity_matrix): 
    """
    Plots a heatmap of provided similarity matrix
    :param similarity_matrix: np.ndarray, pairwise Tanimoto similarity matrix 
    :return: None
    """    
    plt.figure()
    
    plt.imshow(
    similarity_matrix,
    vmin=0,
    vmax=1,
    aspect="auto"
    )


    plt.colorbar(label="Tanimoto similarity")
    plt.xlabel("Molecule Index")
    plt.ylabel("Molecule Index")
    plt.title("Pairwise Tanimoto Similarity")

    plt.show()



Now we have a broad picture of how similar our drug candidates are, but we might want a more informative visualisation. For this we can use dimensionality reduction techniques (see below.)

Note: Up until now we have relied on random methods for our train/test split. For chemistry, there is the possibility that this can lead to similar molecules in training and testing data, especially in real world datasets that contain very similar molecules. This obviously leads to an over-estimation of model performance.

Using Tanimoto similarity to measure the similarity of our training and testing sets can provide a good sanity check to make sure we aren't over estimating our model performance. 

[See here](https://github.com/PatWalters/practical_cheminformatics_posts/blob/main/splitting/dataset_splitting.ipynb) for a more in-depth discussion around dataset splits. 

----

# Dimensionality Reduction

The fingerprints we have generated have 2048 bits, which means that each input has 2048 dimensions. It is common in machine learning to work with high-dimensional datasets, and several methods exist to reduce such dimensionality down to the most important features:
* Principal Component Analysis (PCA)
* t-Distributed Stochastic Neighbour Embedding (tSNE)
* Uniform Manifold Approximation and Projection (UMAP)
* Multi-Dimensional Scaling (MDS).

We will use tSNE to visualise our fingerprints and colour them by predicted activity. 

Use the functions below to perform tSNE dimensionality reduction and visualise your 100 drug candidates 

In [ ]:
from sklearn.manifold import TSNE

def plot_tsne(df):
    """
    Performs t-SNE dimensionality reduction, plots samples and colours by predicted activity
    :param df: pd.DataFrame wih columns "SMILES" and "predicted_activity"
    :return: None
    """
    fps = np.array(get_fingerprints(df["smiles"]))
    
    tsne = TSNE(
        n_components=2, # reduce to 2 dimensions
        random_state=42
    )
    
    embedded_fps = tsne.fit_transform(fps)
    
    scatter = plt.scatter(
        embedded_fps[:, 0], 
        embedded_fps[:, 1],
        c=df["activity"],
    )
    
    plt.colorbar(scatter, label="Predicted activity")

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.title("t-SNE of Molecular Fingerprints")
    
